# Imports


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")


# Filter all the file to obtain just the chosen info (filtri scelti sul portale)

# Filter raws for mutation, sv e gene panel file

In [ ]:
import pandas as pd
import os

def filter_samples_by_id(reference_csv, target_files_list, path, output_prefix="F_"):
    """
    Mantiene nei target_files solo le righe il cui sample_id è presente nel reference_csv.
    """
    
    output_prefix = os.path.join(path, output_prefix)  # Assicurati che il prefisso includa il percorso
    
    # 1. Carica il file di riferimento
    # Uso 'usecols' per caricare solo la colonna che ci serve (risparmia memoria)
    try:
        ref_df = pd.read_csv(reference_csv, sep='\t')
        # Nota: Ho aggiunto .astype(str) per evitare problemi se gli ID sono numerici
        valid_ids = set(ref_df['Sample_Id'].astype(str).unique())
        print(f"Caricati {len(valid_ids)} ID univoci dal file di riferimento.")
    except KeyError:
        print(f"Errore: La colonna 'Sample_Id' non è stata trovata in {reference_csv}")
        return

    # 2. Cicla sui file nel vettore (lista)
    for file_path in target_files_list:
        if not os.path.exists(file_path):
            print(f"File non trovato: {file_path}, salto...")
            continue
            
        print(f"Filtraggio in corso per: {file_path}...")
        
        # Leggi il file target
        target_df = pd.read_csv(file_path, sep='\t')
        
        # Controlla se la colonna esiste (gestendo eventuali differenze di maiuscole/minuscole)
        col_name = 'Sample_Id'

        if col_name in target_df.columns:
            # 3. Filtra: tiene solo le righe dove il sample_id è nel set valid_ids
            filtered_df = target_df[target_df[col_name].astype(str).isin(valid_ids)]
            
            # 4. Salva il risultato
            output_name = f"{output_prefix}{os.path.basename(file_path)}"
            filtered_df.to_csv(output_name, index=False, sep='\t')
            print(f"Salvato: {output_name} ({len(filtered_df)} righe mantenute)")
        else:
            print(f"Colonna ID non trovata in {file_path}. File saltato.")


# --- Tipo tumore ---
# path = "../DHL_prj/kras_colon/"
# master_file = f"{path}F_kras_colon.csv"

path = "../DHL_prj/kras_lung/"
master_file = f"{path}F_kras_lung.csv"

# path = "../DHL_prj/kras_pancreas/"
# master_file = f"{path}F_kras_pancreas.csv"

# Il vettore (lista) degli altri file (es. espressione genica, mutazioni, clinica)
# "data_gene_panel_matrix.csv", "data_gene_panel_matrix.csv", "data_sv.csv", "data_mutations.txt"
altri_file = [f"{path}{filename}" for filename in [ "data_sv.csv", "data_mutations.txt"]]

filter_samples_by_id(master_file, altri_file, path)

Caricati 1233 ID univoci dal file di riferimento.
Filtraggio in corso per: ../DHL_prj/kras_lung/data_gene_panel_matrix.csv...
Salvato: ../DHL_prj/kras_lung/F_data_gene_panel_matrix.csv (1233 righe mantenute)
Filtraggio in corso per: ../DHL_prj/kras_lung/data_sv.csv...
Salvato: ../DHL_prj/kras_lung/F_data_sv.csv (191 righe mantenute)
Filtraggio in corso per: ../DHL_prj/kras_lung/data_mutations.txt...


/var/folders/bk/9ndm0rkn2sv5c7qws_f81k2c0000gn/T/ipykernel_2741/3600799260.py:31: DtypeWarning: Columns (44,49,88) have mixed types. Specify dtype option on import or set low_memory=False.
  target_df = pd.read_csv(file_path, sep='\t')


Salvato: ../DHL_prj/kras_lung/F_data_mutations.txt (11792 righe mantenute)


## Filter the columns of cna file 

In [9]:
def filter_columns_by_master(path, master_path, target_path, output_path):

    master_df = pd.read_csv(f"{path}/{master_path}", sep='\t')
    
    # Crea un set di Sample_Id validi
    valid_ids = set(master_df['Sample_Id'].astype(str).unique())
    print(f"ID univoci trovati nel master: {len(valid_ids)}")

    # 2. Carica il file target (quello da cui levare le colonne)
    target_df = pd.read_csv(f"{path}/{target_path}", sep='\t')
    print(f"Shape del target_df: {target_df.shape}")
    
    # 3. Identifica le colonne da tenere
    # Vogliamo tenere 'Hugo_Symbol' (o altre colonne descrittive iniziali) 
    # E tutte le colonne il cui nome è presente in valid_ids
    columns_to_keep = []
    columns_removed = 0
    
    for col in target_df.columns:
        if col == 'Hugo_Symbol' or col in valid_ids:
            columns_to_keep.append(col)
        else:
            columns_removed += 1
            
    # 4. Filtra il dataframe
    filtered_df = target_df[columns_to_keep]
    
    # 5. Salva il risultato
    filtered_df.to_csv(f"{path}/{output_path}", sep='\t', index=False)
    
    print(f"Colonne totali rimosse: {columns_removed}")
    print(f"Colonne rimaste: {len(columns_to_keep)}")
    print(f"File salvato in: {path}/{output_path}")



path = "../DHL_prj/kras_pancreas/"
filter_columns_by_master(path, 'F_kras_pancreas.csv', 'data_cna.txt', 'F_data_cna.txt')

path = "../DHL_prj/kras_lung/"
filter_columns_by_master(path, 'F_kras_lung.csv', 'data_cna.txt', 'F_data_cna.csv')

path = "../DHL_prj/kras_colon/"
filter_columns_by_master(path, 'F_kras_colon.csv', 'data_cna.txt', 'F_data_cna.csv')

ID univoci trovati nel master: 1233
Shape del target_df: (702, 25035)
Colonne totali rimosse: 23801
Colonne rimaste: 1234
File salvato in: ../DHL_prj/kras_lung//F_data_cna.csv
ID univoci trovati nel master: 1007
Shape del target_df: (702, 25035)
Colonne totali rimosse: 24027
Colonne rimaste: 1008
File salvato in: ../DHL_prj/kras_colon//F_data_cna.csv
